<a href="https://colab.research.google.com/github/yashaskiran/computer_vision_projects/blob/main/object_detection_vehicle_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics roboflow -q

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"Device  : {DEVICE}")

In [ ]:
from ultralytics import YOLO
import cv2, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from pathlib import Path
import math, os, urllib.request

model = YOLO("yolov8n-obb.pt")
model.to(DEVICE)

print("Task    :", model.task)
print("Device  :", DEVICE)
print("Classes :", model.names)

In [ ]:
os.makedirs("test_images", exist_ok=True)

# Use requests with a browser User-Agent to bypass 403 blocks
import requests

def download_image(url, dest):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    try:
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            with open(dest, "wb") as f:
                f.write(r.content)
            return True
        else:
            print(f"  HTTP {r.status_code}")
            return False
    except Exception as e:
        print(f"  Error: {e}")
        return False

IMAGES = {
    # Naval ships — multiple angles (Flickr/public domain sources)
    "naval_ships.jpg"  : "https://live.staticflickr.com/7372/9188691630_7e8c1e2a15_b.jpg",
    # Backup — use the original harbour image which we know works
    "boats_harbour.jpg": "https://ultralytics.com/images/boats.jpg",
}

downloaded = []
for fname, url in IMAGES.items():
    dest = f"test_images/{fname}"
    print(f"Downloading {fname}...")
    if download_image(url, dest):
        downloaded.append(dest)
        print(f"OK   {fname}")
    else:
        print(f"SKIP {fname}")

# Final fallback — generate a synthetic aerial test image if all fail
if not downloaded:
    print("\nAll downloads failed — generating synthetic aerial scene...")
    import numpy as np
    h, w = 800, 800
    synthetic = np.zeros((h, w, 3), dtype=np.uint8)
    synthetic[:] = (30, 60, 30)   # dark green background (water)

    # Draw rotated rectangles simulating ships at various angles
    import random
    random.seed(42)
    for _ in range(25):
        cx  = random.randint(100, 700)
        cy  = random.randint(100, 700)
        ang = random.randint(0, 180)
        l   = random.randint(60, 120)
        w2  = random.randint(15, 25)
        box = cv2.boxPoints(((cx, cy), (l, w2), ang))
        box = np.int32(box)
        cv2.fillPoly(synthetic, [box], (200, 200, 210))
        cv2.polylines(synthetic, [box], True, (255, 255, 255), 1)

    dest = "test_images/synthetic_ships.jpg"
    cv2.imwrite(dest, synthetic)
    downloaded.append(dest)
    print("Generated: synthetic_ships.jpg")

image_files = sorted(Path("test_images").glob("*.jpg"))
print(f"\nImages ready: {len(image_files)}")

# Preview
fig, axes = plt.subplots(1, len(downloaded), figsize=(7*len(downloaded), 6))
if len(downloaded) == 1: axes = [axes]
for ax, p in zip(axes, downloaded):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(Path(p).name, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
CLASS_COLORS = {
    "plane"             : (255, 100,   0),
    "ship"              : (  0, 200, 255),
    "storage-tank"      : (200, 255,   0),
    "large-vehicle"     : (255, 220,  50),
    "small-vehicle"     : (150, 255, 150),
    "helicopter"        : (255,  50, 200),
    "roundabout"        : (100, 100, 255),
    "soccer-ball-field" : ( 50, 255, 100),
    "swimming-pool"     : (100, 200, 255),
    "default"           : (255, 255, 255),
}

def get_color(cls):
    return CLASS_COLORS.get(cls, CLASS_COLORS["default"])

def draw_obb(img, pts_list, label, conf, color):
    pts = np.array(pts_list, dtype=np.int32).reshape((4, 2))
    cv2.polylines(img, [pts], isClosed=True, color=color, thickness=2)
    cx, cy = pts.mean(axis=0).astype(int)
    text = f"{label} {conf:.2f}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(img, (cx-2, cy-th-4), (cx+tw+2, cy+2), color, -1)
    cv2.putText(img, text, (cx, cy-2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)

def run_tiled_obb(model, img_path, tile_size=640, stride=320,
                  conf_thresh=0.25, iou_thresh=0.45):
    full_img = cv2.imread(img_path)
    H, W = full_img.shape[:2]
    annotated = full_img.copy()
    raw_dets = []

    ys = sorted(set(list(range(0, max(1, H-tile_size), stride)) + [max(0, H-tile_size)]))
    xs = sorted(set(list(range(0, max(1, W-tile_size), stride)) + [max(0, W-tile_size)]))
    print(f"  Image {W}x{H} | {len(ys)*len(xs)} tiles ({tile_size}px, stride {stride}px)")

    for y0 in ys:
        y1 = min(y0 + tile_size, H)
        for x0 in xs:
            x1 = min(x0 + tile_size, W)
            tile = full_img[y0:y1, x0:x1]
            res  = model(tile, conf=conf_thresh, iou=iou_thresh,
                         device=DEVICE, verbose=False)[0]
            if res.obb is None or len(res.obb) == 0:
                continue
            obb_xy = res.obb.xyxyxyxy.cpu().numpy()
            c_arr  = res.obb.conf.cpu().numpy()
            cl_arr = res.obb.cls.cpu().numpy().astype(int)
            for i in range(len(c_arr)):
                pts = obb_xy[i].reshape(4, 2).copy()
                pts[:, 0] += x0
                pts[:, 1] += y0
                raw_dets.append({
                    "class" : model.names[cl_arr[i]],
                    "conf"  : float(c_arr[i]),
                    "pts"   : pts.tolist()
                })

    print(f"  Raw detections : {len(raw_dets)}")

    raw_dets.sort(key=lambda d: -d["conf"])
    kept = []
    for det in raw_dets:
        c = np.array(det["pts"]).mean(axis=0)
        dup = any(
            np.linalg.norm(c - np.array(kd["pts"]).mean(axis=0)) < 20
            and det["class"] == kd["class"]
            for kd in kept
        )
        if not dup:
            kept.append(det)

    print(f"  After NMS dedup: {len(kept)}")

    for det in kept:
        draw_obb(annotated, det["pts"], det["class"],
                 det["conf"], get_color(det["class"]))
    return annotated, kept

print("Tiled OBB engine ready.")

In [ ]:
os.makedirs("results", exist_ok=True)
all_results = {}

# CPU users: set stride=640 to skip overlap and run faster
STRIDE = 320 if DEVICE == "cuda" else 640

for img_path in image_files:
    print(f"\n{'='*50}\nProcessing: {img_path.name}")
    annotated, dets = run_tiled_obb(
        model, str(img_path),
        tile_size=640,
        stride=STRIDE,
        conf_thresh=0.25,
        iou_thresh=0.45
    )
    out_path = f"results/{img_path.stem}_obb.jpg"
    cv2.imwrite(out_path, annotated)
    all_results[img_path.name] = {"path": out_path, "dets": dets}
    print(f"  Saved: {out_path}")

print("\nAll images processed!")

In [ ]:
def show_side_by_side(img_name, res_dict):
    orig  = cv2.cvtColor(cv2.imread(f"test_images/{img_name}"), cv2.COLOR_BGR2RGB)
    annot = cv2.cvtColor(cv2.imread(res_dict["path"]), cv2.COLOR_BGR2RGB)
    dets  = res_dict["dets"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(orig);  axes[0].set_title("Original", fontsize=13);  axes[0].axis("off")
    axes[1].imshow(annot); axes[1].set_title(f"OBB Detections ({len(dets)})", fontsize=13)
    axes[1].axis("off")

    legend = [mpatches.Patch(color=np.array(get_color(c))/255, label=c)
              for c in list({d["class"] for d in dets})]
    if legend:
        axes[1].legend(handles=legend, loc="lower right", fontsize=8, framealpha=0.7)

    plt.suptitle(img_name, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

for img_name, res in all_results.items():
    show_side_by_side(img_name, res)

In [ ]:
from collections import Counter, defaultdict

# ── Use only the harbour/dock image ─────────────────────────────────────────
DOCK_IMAGE = "boats_harbour.jpg"

if DOCK_IMAGE not in all_results:
    print(f"Running inference on {DOCK_IMAGE}...")
    img_path = Path(f"test_images/{DOCK_IMAGE}")
    annotated, dets = run_tiled_obb(
        model, str(img_path),
        tile_size=640, stride=STRIDE,
        conf_thresh=0.25, iou_thresh=0.45
    )
    out_path = f"results/{img_path.stem}_obb.jpg"
    cv2.imwrite(out_path, annotated)
    all_results[DOCK_IMAGE] = {"path": out_path, "dets": dets}

all_dets_flat = all_results[DOCK_IMAGE]["dets"]
print(f"Total detections in dock image: {len(all_dets_flat)}")

# ── Compute OBB tightness ratios ─────────────────────────────────────────────
ROTATION_CLASSES = {"ship", "plane", "helicopter", "large-vehicle", "harbor"}

ratios_all      = []
ratios_filtered = []

for d in all_dets_flat:
    pts = np.array(d["pts"], dtype=np.float32)
    obb_area  = cv2.contourArea(pts)
    x, y, w, h = cv2.boundingRect(pts.astype(np.int32))
    aabb_area = w * h if w * h > 0 else 1
    if obb_area > 0:
        ratio = obb_area / aabb_area
        ratios_all.append(ratio)
        if d["class"] in ROTATION_CLASSES:
            ratios_filtered.append(ratio)

saved_all      = (1 - np.mean(ratios_all))      * 100 if ratios_all      else 0
saved_filtered = (1 - np.mean(ratios_filtered)) * 100 if ratios_filtered else 0

# ── Per-class breakdown ───────────────────────────────────────────────────────
class_ratios = defaultdict(list)
for d in all_dets_flat:
    pts = np.array(d["pts"], dtype=np.float32)
    obb_area  = cv2.contourArea(pts)
    x, y, w, h = cv2.boundingRect(pts.astype(np.int32))
    aabb_area = w * h if w * h > 0 else 1
    if obb_area > 0:
        class_ratios[d["class"]].append(obb_area / aabb_area)

class_names  = list(class_ratios.keys())
mean_savings = [(1 - np.mean(class_ratios[c])) * 100 for c in class_names]
counts       = [len(class_ratios[c]) for c in class_names]

sorted_idx   = np.argsort(mean_savings)[::-1]
class_names  = [class_names[i] for i in sorted_idx]
mean_savings = [mean_savings[i] for i in sorted_idx]
counts       = [counts[i] for i in sorted_idx]

# ── 4-panel dashboard ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 10))
fig.suptitle("OBB Detection Statistics — Harbour/Dock Image",
             fontsize=15, fontweight="bold")

# Panel 1 — Class distribution
ax1 = fig.add_subplot(2, 3, 1)
class_counts = Counter(d["class"] for d in all_dets_flat)
labels = list(class_counts.keys())
vals   = list(class_counts.values())
ax1.barh(labels, vals, color=[np.array(get_color(l))/255 for l in labels])
ax1.set_xlabel("Count"); ax1.set_title("Detections by Class")
for i, v in enumerate(vals):
    ax1.text(v+0.3, i, str(v), va="center", fontsize=8)

# Panel 2 — Confidence histogram
ax2 = fig.add_subplot(2, 3, 2)
confs = [d["conf"] for d in all_dets_flat]
ax2.hist(confs, bins=25, color="#4f98a3", edgecolor="white")
ax2.axvline(np.mean(confs), color="orange", lw=2,
            label=f"mean={np.mean(confs):.2f}")
ax2.set_xlabel("Confidence"); ax2.set_ylabel("Count")
ax2.set_title("Confidence Distribution"); ax2.legend()

# Panel 3 — All vs rotation-rich OBB tightness overlaid
ax3 = fig.add_subplot(2, 3, 3)
ax3.hist(ratios_all, bins=25, color="#a86fdf", edgecolor="white",
         alpha=0.6, label=f"All classes ({saved_all:.1f}% saved)")
if ratios_filtered:
    ax3.hist(ratios_filtered, bins=25, color="#4f98a3", edgecolor="white",
             alpha=0.6, label=f"Rotated classes ({saved_filtered:.1f}% saved)")
ax3.axvline(np.mean(ratios_all), color="#a86fdf", lw=2, linestyle="--")
if ratios_filtered:
    ax3.axvline(np.mean(ratios_filtered), color="#4f98a3", lw=2, linestyle="--")
ax3.set_xlabel("OBB area / AABB area  (lower = more savings)")
ax3.set_ylabel("Count")
ax3.set_title("OBB Tightness: All vs Rotated Classes")
ax3.legend(fontsize=8)

# Panel 4 — Per-class savings bar chart
ax4 = fig.add_subplot(2, 1, 2)
ax4.barh(class_names, mean_savings,
         color=[np.array(get_color(c))/255 for c in class_names],
         edgecolor="white")
ax4.axvline(saved_all, color="orange", lw=2, linestyle="--",
            label=f"Overall mean {saved_all:.1f}%")
ax4.set_xlabel("AABB Wasted Area Saved by OBB (%)")
ax4.set_title("OBB Advantage per Class — Higher = More Rotation Savings",
              fontsize=12)
ax4.set_xlim(0, 100)
for i, (v, n) in enumerate(zip(mean_savings, counts)):
    ax4.text(v + 0.8, i, f"{v:.1f}%  (n={n})", va="center", fontsize=9)
ax4.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── Terminal summary ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"IMAGE             : {DOCK_IMAGE}")
print(f"Total detections  : {len(all_dets_flat)}")
print(f"All-class saving  : {saved_all:.1f}%")
print(f"Rotated-class     : {saved_filtered:.1f}%")
print(f"Mean confidence   : {np.mean(confs):.3f}")
print("\nPer-class breakdown:")
for c, s, n in zip(class_names, mean_savings, counts):
    bar = "█" * int(s / 5)
    print(f"  {c:<22} {bar:<20} {s:.1f}%  (n={n})")

In [ ]:
# Add this after Stage 5C angle analysis
# Recompute savings filtering out near-axis-aligned boxes

ANGLE_THRESHOLD = 15  # ignore boxes within 15 deg of horizontal/vertical

high_rotation_dets = []
for img_path in image_files:
    res = model(str(img_path), conf=0.25, iou=0.45, device=DEVICE, verbose=False)[0]
    if res.obb is None or len(res.obb) == 0:
        continue
    xywhr  = res.obb.xywhr.cpu().numpy()
    obb_xy = res.obb.xyxyxyxy.cpu().numpy()
    confs  = res.obb.conf.cpu().numpy()
    clses  = res.obb.cls.cpu().numpy().astype(int)

    for i in range(len(confs)):
        angle_deg = math.degrees(xywhr[i, 4]) % 180
        # Keep only genuinely rotated boxes (not near 0, 90, 180)
        not_axis_aligned = (
            ANGLE_THRESHOLD < angle_deg < (90 - ANGLE_THRESHOLD) or
            (90 + ANGLE_THRESHOLD) < angle_deg < (180 - ANGLE_THRESHOLD)
        )
        if not_axis_aligned:
            pts = obb_xy[i].reshape(4, 2)
            obb_area  = cv2.contourArea(pts.astype(np.float32))
            x, y, w, h = cv2.boundingRect(pts.astype(np.int32))
            aabb_area = w * h if w * h > 0 else 1
            if obb_area > 0:
                high_rotation_dets.append({
                    "class"    : model.names[clses[i]],
                    "ratio"    : obb_area / aabb_area,
                    "angle_deg": angle_deg,
                    "conf"     : float(confs[i])
                })

if high_rotation_dets:
    ratios = [d["ratio"] for d in high_rotation_dets]
    saving = (1 - np.mean(ratios)) * 100
    print(f"\nRotated-only detections  : {len(high_rotation_dets)}")
    print(f"OBB saving (rotated only): {saving:.1f}%")
    print(f"(Excluded axis-aligned boxes within {ANGLE_THRESHOLD} deg of 0/90/180)")

In [ ]:
# Run this after Stage 5C

angle_saving_data = []
for img_path in image_files:
    res = model(str(img_path), conf=0.25, iou=0.45, device=DEVICE, verbose=False)[0]
    if res.obb is None or len(res.obb) == 0:
        continue
    xywhr  = res.obb.xywhr.cpu().numpy()
    obb_xy = res.obb.xyxyxyxy.cpu().numpy()
    clses  = res.obb.cls.cpu().numpy().astype(int)

    for i in range(len(res.obb)):
        angle_deg = math.degrees(xywhr[i, 4]) % 90  # fold to 0-90
        pts = obb_xy[i].reshape(4, 2)
        obb_area  = cv2.contourArea(pts.astype(np.float32))
        x, y, w, h = cv2.boundingRect(pts.astype(np.int32))
        aabb_area = w * h if w * h > 0 else 1
        if obb_area > 0 and aabb_area > 0:
            saving_pct = (1 - obb_area / aabb_area) * 100
            angle_saving_data.append({
                "angle_deg"  : angle_deg,
                "saving_pct" : saving_pct,
                "class"      : model.names[clses[i]]
            })

if angle_saving_data:
    angles  = [d["angle_deg"]   for d in angle_saving_data]
    savings = [d["saving_pct"]  for d in angle_saving_data]
    classes = [d["class"]       for d in angle_saving_data]

    # Theoretical max saving curve: 1 - (pi/4) * sin(2*theta)
    theta_range = np.linspace(0, 90, 200)
    theoretical = [abs(math.sin(math.radians(2*t))) * 50 for t in theta_range]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("OBB Saving vs Rotation Angle", fontsize=13, fontweight="bold")

    # Scatter: actual saving per detection
    uc = list(set(classes))
    cm = plt.cm.Set2(np.linspace(0, 1, len(uc)))
    for ci, c in enumerate(uc):
        idx = [i for i, x in enumerate(classes) if x == c]
        axes[0].scatter(
            [angles[i] for i in idx],
            [savings[i] for i in idx],
            label=c, color=cm[ci], alpha=0.5, s=20, edgecolors="none"
        )
    axes[0].plot(theta_range, theoretical, "orange", lw=2,
                 linestyle="--", label="Theoretical max (rectangle)")
    axes[0].set_xlabel("OBB Angle folded to 0-90 deg")
    axes[0].set_ylabel("Area Saved vs AABB (%)")
    axes[0].set_title("Actual Saving per Detection")
    axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

    # Binned mean saving curve
    bin_edges  = np.linspace(0, 90, 19)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_means  = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        vals = [s for a, s in zip(angles, savings) if lo <= a < hi]
        bin_means.append(np.mean(vals) if vals else 0)

    axes[1].bar(bin_centers, bin_means, width=5, color="#4f98a3",
                edgecolor="white", alpha=0.85, label="Mean saving per bin")
    axes[1].plot(theta_range, theoretical, "orange", lw=2,
                 linestyle="--", label="Theoretical max")
    axes[1].set_xlabel("OBB Angle (deg)")
    axes[1].set_ylabel("Mean Area Saved (%)")
    axes[1].set_title("Binned Mean Saving by Angle")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()

    print(f"\nPeak saving angle  : {bin_centers[np.argmax(bin_means)]:.0f} deg")
    print(f"Peak saving value  : {max(bin_means):.1f}%")
    print(f"Detections at 45deg: {sum(1 for a in angles if 40 <= a <= 50)}")

In [ ]:
angle_data = []
for img_path in image_files:
    res = model(str(img_path), conf=0.25, iou=0.45,
                device=DEVICE, verbose=False)[0]
    if res.obb is None or len(res.obb) == 0:
        continue
    xywhr = res.obb.xywhr.cpu().numpy()
    confs  = res.obb.conf.cpu().numpy()
    clses  = res.obb.cls.cpu().numpy().astype(int)
    for i in range(len(confs)):
        angle_data.append({
            "class"     : model.names[clses[i]],
            "conf"      : float(confs[i]),
            "angle_deg" : math.degrees(xywhr[i, 4]) % 180,
            "w"         : float(xywhr[i, 2]),
            "h"         : float(xywhr[i, 3])
        })

if angle_data:
    fig = plt.figure(figsize=(14, 5))
    fig.suptitle("Rotation Angle Analysis", fontsize=13, fontweight="bold")

    ax_p = fig.add_subplot(121, polar=True)
    ang_rad = [math.radians(d["angle_deg"]) for d in angle_data]
    bins = np.linspace(0, np.pi, 19)
    hist, edges = np.histogram(ang_rad, bins=bins)
    ax_p.bar((edges[:-1]+edges[1:])/2, hist, width=np.diff(edges),
             color="#4f98a3", alpha=0.85, edgecolor="white")
    ax_p.set_title("OBB Orientations (0-180 deg)", pad=15, fontsize=10)

    ax2 = fig.add_subplot(122)
    area = [d["w"]*d["h"] for d in angle_data]
    ang  = [d["angle_deg"] for d in angle_data]
    cls  = [d["class"] for d in angle_data]
    uc   = list(set(cls))
    cm   = plt.cm.Set2(np.linspace(0, 1, len(uc)))
    for ci, c in enumerate(uc):
        idx = [i for i, x in enumerate(cls) if x == c]
        ax2.scatter([ang[i] for i in idx], [area[i] for i in idx],
                    label=c, color=cm[ci], alpha=0.8, s=60,
                    edgecolors="white", lw=0.5)
    ax2.set_xlabel("Rotation Angle (deg)")
    ax2.set_ylabel("OBB Area (px squared)")
    ax2.set_title("Object Size vs Rotation Angle")
    ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"{len(angle_data)} detections | "
          f"Angles: {min(ang):.1f} - {max(ang):.1f} deg | Mean: {np.mean(ang):.1f} deg")
else:
    print("No detections — try conf=0.15")

In [ ]:
# ── Filter to only the two diagonal clusters ─────────────────────────────────
# Cluster 1: 10–30 deg (shallow diagonal ships)
# Cluster 2: 50–70 deg (steeper diagonal ships)

CLUSTER_1 = (10, 30)
CLUSTER_2 = (50, 70)

diagonal_dets = []

for img_path in image_files:
    res = model(str(img_path), conf=0.25, iou=0.45, device=DEVICE, verbose=False)[0]
    if res.obb is None or len(res.obb) == 0:
        continue
    xywhr  = res.obb.xywhr.cpu().numpy()
    obb_xy = res.obb.xyxyxyxy.cpu().numpy()
    confs  = res.obb.conf.cpu().numpy()
    clses  = res.obb.cls.cpu().numpy().astype(int)

    for i in range(len(confs)):
        angle_deg = math.degrees(xywhr[i, 4]) % 90  # fold to 0-90

        in_cluster1 = CLUSTER_1[0] <= angle_deg <= CLUSTER_1[1]
        in_cluster2 = CLUSTER_2[0] <= angle_deg <= CLUSTER_2[1]

        if not (in_cluster1 or in_cluster2):
            continue

        pts = obb_xy[i].reshape(4, 2)
        obb_area  = cv2.contourArea(pts.astype(np.float32))
        x, y, w, h = cv2.boundingRect(pts.astype(np.int32))
        aabb_area = w * h if w * h > 0 else 1

        if obb_area > 0:
            saving_pct = (1 - obb_area / aabb_area) * 100
            diagonal_dets.append({
                "class"     : model.names[clses[i]],
                "conf"      : float(confs[i]),
                "angle_deg" : angle_deg,
                "saving_pct": saving_pct,
                "cluster"   : "10-30 deg" if in_cluster1 else "50-70 deg"
            })

# ── Print summary ─────────────────────────────────────────────────────────────
c1 = [d for d in diagonal_dets if d["cluster"] == "10-30 deg"]
c2 = [d for d in diagonal_dets if d["cluster"] == "50-70 deg"]

print("="*55)
print(f"Total diagonal detections : {len(diagonal_dets)}")
print(f"Cluster 1 (10-30 deg)     : {len(c1)} detections")
if c1: print(f"  Mean saving             : {np.mean([d['saving_pct'] for d in c1]):.1f}%")
if c1: print(f"  Max saving              : {max([d['saving_pct'] for d in c1]):.1f}%")
print(f"Cluster 2 (50-70 deg)     : {len(c2)} detections")
if c2: print(f"  Mean saving             : {np.mean([d['saving_pct'] for d in c2]):.1f}%")
if c2: print(f"  Max saving              : {max([d['saving_pct'] for d in c2]):.1f}%")
overall_saving = np.mean([d["saving_pct"] for d in diagonal_dets])
print(f"\nOverall diagonal saving   : {overall_saving:.1f}%")
print(f"(vs 48.9% when axis-aligned boats included)")
print("="*55)

# ── Plot filtered results ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("OBB Savings — Diagonal Ship Clusters Only", fontsize=13, fontweight="bold")

colors = {"10-30 deg": "#4f98a3", "50-70 deg": "#a86fdf"}

# Left — scatter per detection coloured by cluster
for cluster, color in colors.items():
    pts = [d for d in diagonal_dets if d["cluster"] == cluster]
    if pts:
        axes[0].scatter(
            [d["angle_deg"] for d in pts],
            [d["saving_pct"] for d in pts],
            label=f"{cluster}  (mean {np.mean([d['saving_pct'] for d in pts]):.1f}%)",
            color=color, alpha=0.75, s=50, edgecolors="white", lw=0.5
        )

# Theoretical max curve
theta_range  = np.linspace(0, 90, 200)
theoretical  = [abs(math.sin(math.radians(2*t))) * 50 for t in theta_range]
axes[0].plot(theta_range, theoretical, "orange", lw=2,
             linestyle="--", label="Theoretical max (rectangle)")
axes[0].axhline(overall_saving, color="red", lw=1.5, linestyle=":",
                label=f"Diagonal mean {overall_saving:.1f}%")
axes[0].set_xlabel("OBB Angle (deg, folded to 0-90)")
axes[0].set_ylabel("Area Saved vs AABB (%)")
axes[0].set_title("Actual Saving — Diagonal Clusters Only")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, 90); axes[0].set_ylim(0, 80)

# Right — box plot comparing the two clusters
c1_savings = [d["saving_pct"] for d in c1]
c2_savings = [d["saving_pct"] for d in c2]

bp = axes[1].boxplot(
    [c1_savings, c2_savings],
    labels=["Cluster 1\n(10-30 deg)", "Cluster 2\n(50-70 deg)"],
    patch_artist=True,
    medianprops=dict(color="orange", lw=2)
)
bp["boxes"][0].set_facecolor("#4f98a3")
bp["boxes"][1].set_facecolor("#a86fdf")
for box in bp["boxes"]: box.set_alpha(0.75)

axes[1].axhline(48.9, color="red", lw=1.5, linestyle=":",
                label="Previous overall mean 48.9%")
axes[1].axhline(overall_saving, color="green", lw=1.5, linestyle="--",
                label=f"Diagonal mean {overall_saving:.1f}%")
axes[1].set_ylabel("Area Saved vs AABB (%)")
axes[1].set_title("Distribution per Cluster")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis="y")
axes[1].set_ylim(0, 80)

plt.tight_layout()
plt.show()

In [ ]:
# Cleaner single-cluster filter — captures all genuinely rotated ships
# Exclude only the near-axis-aligned outliers (0-10 deg and 80-90 deg)

ROTATED_MIN = 10   # exclude 0-10 deg (axis-aligned)
ROTATED_MAX = 80   # exclude 80-90 deg (axis-aligned the other way)

rotated_dets = [
    d for d in diagonal_dets
    if ROTATED_MIN <= d["angle_deg"] <= ROTATED_MAX
]

savings = [d["saving_pct"] for d in rotated_dets]

print(f"Rotated detections       : {len(rotated_dets)}")
print(f"Mean OBB saving          : {np.mean(savings):.1f}%")
print(f"Median OBB saving        : {np.median(savings):.1f}%")
print(f"Max OBB saving           : {max(savings):.1f}%")
print(f"Detections above 50%     : {sum(1 for s in savings if s >= 50)}  "
      f"({sum(1 for s in savings if s >= 50)/len(savings)*100:.0f}% of total)")
print(f"Detections above 60%     : {sum(1 for s in savings if s >= 60)}  "
      f"({sum(1 for s in savings if s >= 60)/len(savings)*100:.0f}% of total)")

In [ ]:
def detect_and_show(image_path, conf=0.25, tile=True):
    if tile:
        annotated, dets = run_tiled_obb(model, image_path, conf_thresh=conf)
    else:
        res = model(image_path, conf=conf, device=DEVICE, verbose=False)
        annotated = res[0].plot()
        r = res[0]; dets = []
        if r.obb is not None:
            for i in range(len(r.obb)):
                dets.append({
                    "class" : model.names[int(r.obb.cls[i])],
                    "conf"  : float(r.obb.conf[i])
                })

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{Path(image_path).name}  |  {len(dets)} OBBs  |  conf>={conf}")
    ax.axis("off"); plt.tight_layout(); plt.show()
    return dets

# Run on first image
if image_files:
    dets = detect_and_show(str(image_files[0]), conf=0.20, tile=True)
    for d in dets[:10]:
        print(f"  {d['class']:<22}  conf={d['conf']:.3f}")

In [ ]:
print("Exporting to ONNX...")
export_path = model.export(
    format   = "onnx",
    imgsz    = 640,
    simplify = True,
    dynamic  = False,
    opset    = 17,
)
print(f"ONNX saved: {export_path}")
print("Next: drop into TensorRT FP16 for 5-10x speedup on edge hardware.")